<a href="https://colab.research.google.com/github/Uyhuynh23/Group13_CV/blob/main/notebooks/colab_flickr8k_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flickr8k Baseline Runner (Google Colab + Google Drive + GitHub)
**Project:** *A Controlled Evaluation of Patch-Preserving and Query-Resampling Connectors for Resource-Constrained Image Captioning*  
**Group 13:** Huỳnh Trần Uy (23125047), Trần Mỹ An (23125049), Nguyễn Hoàng Gia Hân (23125055)

### Workflow
1. **Code** lives on GitHub (`https://github.com/Uyhuynh23/Group13_CV.git`). Whenever you edit code locally and `git push`, just re-run **Step 2** (`!git pull`) in this notebook.
2. **Data, Cached CLIP Tensors, Checkpoints, and Evaluation Logs** live on your **Google Drive** (`/content/drive/MyDrive/Group13_CV_Data`) so you never lose progress when Colab disconnects.

## Step 1: Mount Google Drive & Verify Tesla T4 GPU

In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create persistent directories on Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/Group13_CV_Data'
ZIP_CACHE_DIR = f'{DRIVE_ROOT}/flickr8k_zips'
CLIP_CACHE_DIR = f'{DRIVE_ROOT}/cache'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
LOCAL_DATA_DIR = '/content/flickr8k_local'

for d in [DRIVE_ROOT, ZIP_CACHE_DIR, CLIP_CACHE_DIR, CKPT_DIR, RESULTS_DIR, LOCAL_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Persistent Google Drive root:', DRIVE_ROOT)
!nvidia-smi

Mounted at /content/drive
Persistent Google Drive root: /content/drive/MyDrive/Group13_CV_Data
/bin/bash: line 1: nvidia-smi: command not found


## Step 2: Clone / Pull GitHub Repository & Install Dependencies
Re-run the cell below anytime you push code updates to GitHub.

In [4]:
REPO_URL = 'https://github.com/Uyhuynh23/Group13_CV.git'
REPO_DIR = '/content/Group13_CV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!pip install -q -r requirements.txt

Cloning into '/content/Group13_CV'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 186 (delta 81), reused 92 (delta 74), pack-reused 76 (from 1)
Receiving objects: 100% (186/186), 2.30 MiB | 13.35 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/content/Group13_CV
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Step 3: Auto-Download Flickr8k to Google Drive & Cache CLIP ViT-B/32 Features
- Automatically downloads `Flickr8k_Dataset.zip` (1.1 GB) and `Flickr8k_text.zip` directly into your Google Drive (`flickr8k_zips/`) so you never have to upload 1 GB manually.
- Extracts both **1×512 global CLIP features** (for the sanity baseline) and **49×768 patch tokens in FP16** (~0.60 GB total, for the upcoming 2×2 study) into `train_cache.pt`, `val_cache.pt`, and `test_cache.pt` on Google Drive.
- **Note:** Once `train_cache.pt`, `val_cache.pt`, and `test_cache.pt` exist on your Google Drive, future sessions skip downloading and feature extraction automatically!

In [ ]:
%cd /content/Group13_CV

expected_caches = [
    f'{CLIP_CACHE_DIR}/train_cache.pt',
    f'{CLIP_CACHE_DIR}/val_cache.pt',
    f'{CLIP_CACHE_DIR}/test_cache.pt',
]

if all(os.path.isfile(p) for p in expected_caches):
    print('[SKIP] All CLIP feature caches already exist on Google Drive!')
    for p in expected_caches:
        print(f'  - {p} ({os.path.getsize(p)/(1024**2):.1f} MB)')
else:
    # 1. Download Flickr8k zips to Google Drive and extract locally for fast I/O
    !python download_flickr8k.py --data_dir {LOCAL_DATA_DIR} --zip_cache_dir {ZIP_CACHE_DIR}

    # 2. Extract and cache frozen CLIP ViT-B/32 features (FP16) onto Google Drive
    !python parse_flickr8k.py \
        --data_dir {LOCAL_DATA_DIR} \
        --splits_dir ./data/flickr8k/splits \
        --cache_dir {CLIP_CACHE_DIR} \
        --batch_size 64

## Step 4: Train the Pipeline Sanity Baseline (`global_linear`)
Implements Proposal Section IV.D & IV.E:
- **Architecture:** Global CLIP ViT-B/32 feature (`512`-d) + Linear projection to GPT-2 prefix (`1 × 768`), with frozen GPT-2 Small decoder.
- **Checkpoint selection:** Saved to Google Drive (`checkpoints/baseline_global_linear_seed42_best.pt`) based on validation CIDEr.

In [ ]:
%cd /content/Group13_CV

!python train_flickr8k.py \
    --cache_dir {CLIP_CACHE_DIR} \
    --out_dir {CKPT_DIR} \
    --run_name baseline_global_linear_seed42 \
    --connector_type global_linear \
    --prefix_length 1 \
    --epochs 10 \
    --bs 64 \
    --lr 2e-4 \
    --patience 4 \
    --seed 42

### (Optional) Train Original ClipCap MLP Baseline (`clipcap_mlp`, `prefix_length=10`)

In [ ]:
# Uncomment to also run the 10-token ClipCap MLP baseline:
# !python train_flickr8k.py \
#     --cache_dir {CLIP_CACHE_DIR} \
#     --out_dir {CKPT_DIR} \
#     --run_name baseline_clipcap_mlp_seed42 \
#     --connector_type clipcap_mlp \
#     --prefix_length 10 \
#     --epochs 10 \
#     --bs 64 \
#     --lr 2e-4 \
#     --patience 4 \
#     --seed 42

## Step 5: Evaluate Baseline on Flickr8k Test Split (1,000 Images)
Computes **CIDEr, BLEU-1, BLEU-4, METEOR, ROUGE-L**, plus decoding latency (`ms/image`) and `tokens/sec` using deterministic Beam Search (`beam_size=5`) and Greedy Decoding (`beam_size=1`).

In [5]:
%cd /content/Group13_CV

BEST_CKPT = f'{CKPT_DIR}/baseline_global_linear_seed42_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

# 1. Deterministic Beam Search (beam_size = 5)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

# 2. Greedy Decoding Robustness Check (beam_size = 1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

/content/Group13_CV
Loading checkpoint from /content/drive/MyDrive/Group13_CV_Data/checkpoints/baseline_global_linear_seed42_best.pt on cpu ...
config.json: 100% 665/665 [00:00<00:00, 2.12MB/s]

model.safetensors: downloading bytes:  46% 255M/548M [00:01<00:01, 225MB/s, 22.2MB/s  ]
model.safetensors: downloading bytes:  84% 462M/548M [00:02<00:00, 260MB/s, 38.3MB/s  ]
model.safetensors: reconstructing file:  61% 335M/548M [00:02<00:01, 110MB/s, 19.2MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:05<00:00, 87.3MB/s, 41.9MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:05<00:00, 101MB/s, 42.4MB/s  ] 
Loading weights: 100% 148/148 [00:00<00:00, 4173.63it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 470kB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 107kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 20.0MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 29.7MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 38.4MB/s]
Evaluating test (bea

## Step 6: Inspect Qualitative Predictions & Training Summary

In [6]:
import json

eval_file = f'{RESULTS_DIR}/eval_test_baseline_global_linear_seed42_best_beam5.json'
with open(eval_file, 'r') as f:
    report = json.load(f)

print('=== SAMPLE GENERATED CAPTIONS (TEST SET) ===\n')
for item in report['predictions'][:5]:
    print(f"Image ID   : {item['image_id']}")
    print(f"Prediction : {item['prediction']}")
    print(f"Reference 1: {item['references'][0]}")
    print(f"Reference 2: {item['references'][1]}")
    print('-' * 60)

=== SAMPLE GENERATED CAPTIONS (TEST SET) ===

Image ID   : 3385593926_d3e9c21170.jpg
Prediction : Two dogs are playing in the grass .
Reference 1: The dogs are in the snow in front of a fence .
Reference 2: The dogs play on the snow .
------------------------------------------------------------
Image ID   : 2677656448_6b7e7702af.jpg
Prediction : A dog is swimming in a pool .
Reference 1: a brown and white dog swimming towards some in the pool
Reference 2: A dog in a swimming pool swims toward sombody we cannot see .
------------------------------------------------------------
Image ID   : 311146855_0b65fdb169.jpg
Prediction : A man and a woman are playing in the street .
Reference 1: A man and a woman in festive costumes dancing .
Reference 2: A man and a woman with feathers on her head dance .
------------------------------------------------------------
Image ID   : 1258913059_07c613f7ff.jpg
Prediction : A man and woman are sitting on a bench in front of a bar .
Reference 1: A couple 